In [12]:
import urllib.request
import json
import pymongo


In [13]:
# get the bbc rss feed of news stories and connect to it
earthquake_url = "http://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/significant_month.geojson"

try:
    response = urllib.request.urlopen(earthquake_url)
except urllib.error.URLError as e:
    if hasattr(e, 'reason'):
        print('We failed to reach a server.')
        print('Reason: ', e.reason)
    elif hasattr(e, 'code'):
        print('The server couldn\'t fulfill the request.')
        print('Error code: ', e.code)
else:
    # the url request was successful - convert the response to a string
    json_string = response.read().decode('utf-8')

In [14]:
# the json package loads() converts the string to python dictionaries and lists
eq_json = json.loads(json_string)

# from the json dictionary we get the title to print
title = eq_json['metadata']['title']
print('Collected data from', title)
#  and we get the list of earthquakes
quakelist = eq_json['features']


Collected data from USGS Significant Earthquakes, Past Month


In [15]:
# Connection to Mongo DB
try:
    client=pymongo.MongoClient('localhost', 27017)
    print ("Connected successfully!!!")
except pymongo.errors.ConnectionFailure as e:
   print ("Could not connect to MongoDB: %s" % e )
else:
    # use database named usgs or create it if not there already
    eqdb = client.usgs
    # create collection named earthquakes or create it if not there already
    quakecoll = eqdb.earthquakes
    # add all the earthquakes to the list
    quakecoll.insert_many(quakelist)
    print("Added", len(quakelist), "to earthquakes collection in usgs database")
    

Connected successfully!!!
Added 12 to earthquakes collection in usgs database


In [16]:
#Listing all dbs in the client
client.list_database_names()

['admin', 'config', 'local', 'sampledb', 'usgs']

In [17]:
# Setting the usgs db to a variable and then seeing the collections 
db = client.usgs
db.list_collection_names()

['earthquakes']

In [21]:
#Generate a list of earthquakes. BOOM!

quakelist = db.earthquakes
docs = quakelist.find({}, {'_id': 1, 'properties.place': 1, 'properties.time': 1})
for doc in docs:
    print(doc)
    

{'_id': ObjectId('655f5b92251e29e5afaad21c'), 'properties': {'place': '96 km E of Port-Olry, Vanuatu', 'time': 1700628451605}}
{'_id': ObjectId('655f5b92251e29e5afaad21d'), 'properties': {'place': '26 km WSW of Burias, Philippines', 'time': 1700208853236}}
{'_id': ObjectId('655f5b92251e29e5afaad21e'), 'properties': {'place': 'Banda Sea', 'time': 1699448526600}}
{'_id': ObjectId('655f5b92251e29e5afaad21f'), 'properties': {'place': 'Coalson Draw, Texas', 'time': 1699439269034}}
{'_id': ObjectId('655f5b92251e29e5afaad220'), 'properties': {'place': 'Banda Sea', 'time': 1699419230459}}
{'_id': ObjectId('655f5b92251e29e5afaad221'), 'properties': {'place': 'Banda Sea', 'time': 1699419171835}}
{'_id': ObjectId('655f5b92251e29e5afaad222'), 'properties': {'place': '43 km E of Dailekh, Nepal', 'time': 1699034576557}}
{'_id': ObjectId('655f5b92251e29e5afaad223'), 'properties': {'place': '81 km WSW of Vallenar, Chile', 'time': 1698755623935}}
{'_id': ObjectId('655f5b92251e29e5afaad224'), 'propertie

In [22]:
client.close()